# RG3 점진 축소
저장된 구간 제외 예측으로 가지 제거 비교. 독립 최종 검증 아님.

In [1]:
from pathlib import Path
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
tables=root/'reports/tables'
p=pd.read_csv(tables/'rg3_pressure_seed_majority_predictions.csv')
p=p[(p.scheme=='leave_one_block_out') & (p.anchor=='exploratory54')].sort_values('TimeStamp').reset_index(drop=True)
assert len(p)==583 and p.actual.sum()==17 and p.TimeStamp.is_unique
i=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv')
i=i[(i.scheme=='leave_one_block_out') & (i.model=='three_seed_mean')].set_index('TimeStamp').loc[p.TimeStamp]
t=pd.read_csv(tables/'rg3_pdf_confirm_gates_predictions.csv')
t=t[(t.scheme=='leave_one_block_out') & (t.model=='thermal_change_low_0.05')].set_index('TimeStamp').loc[p.TimeStamp]
baseline=i.baseline_prediction.to_numpy(bool)
innovation=(i.expert_score.to_numpy()>i.expert_threshold.to_numpy()) & (i.current_barrel5_delta.to_numpy()>i.delta_threshold.to_numpy())
thermal=(t.score.to_numpy()>t.score_threshold.to_numpy()) & t.gate_pass.to_numpy(bool)
assert np.array_equal(baseline|innovation,i.prediction.to_numpy(bool))
assert np.array_equal(baseline|innovation|thermal,p.anchor_prediction.to_numpy(bool))
seedcols=['seed42_7_101','seed11_23_37','seed13_29_53']
components=pd.read_csv(tables/'rg3_pressure_seed_predictions.csv')
components=components[(components.scheme=='leave_one_block_out') & (components.anchor=='exploratory54') & (components.model=='pressure_fall_speed_rise_switch_steeper')]
for tag in seedcols:
 c=components[components.configuration.eq(tag)].set_index('TimeStamp').loc[p.TimeStamp]
 p[tag]=((c.score.to_numpy()>c.threshold.to_numpy()) & c.direction_pass.to_numpy(bool)).astype(int)
assert np.array_equal(p.anchor_prediction.to_numpy(bool)|(p[seedcols].sum(axis=1).to_numpy()>=2),p.prediction.to_numpy(bool))
rows=[];preds=[]
for use_i,use_t in [(True,True),(False,True),(True,False),(False,False)]:
 for n in [3,2,1,0]:
  for ids in itertools.combinations(seedcols,n):
   rules=[2] if n==3 else ([1,2] if n==2 else [1])
   for votes in rules:
    pressure=p[list(ids)].sum(axis=1).to_numpy()>=votes if n else np.zeros(len(p),bool)
    pr=baseline|(innovation if use_i else False)|(thermal if use_t else False)|pressure
    count=2+12*use_i+9*use_t+9*n
    name=f'I{int(use_i)}_T{int(use_t)}_'+('_'.join(ids) or 'no_pressure')+f'_vote{votes}'
    tn,fp,fn,tp=confusion_matrix(p.actual,pr,labels=[0,1]).ravel()
    rows.append(dict(candidate=name,classifiers=count,innovation=use_i,thermal=use_t,pressure_groups=n,votes=votes,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn)))
    preds.append(p[['TimeStamp','actual']].assign(candidate=name,prediction=pr.astype(int)))
metrics=pd.DataFrame(rows)
original=metrics[(metrics.classifiers==50)]
assert len(original)==1 and original.iloc[0].TP==8 and original.iloc[0].FP==6
metrics.to_csv(tables/'rg3_190_gradual_branch_ablation_metrics.csv',index=False,encoding='utf-8-sig')
pd.concat(preds).to_csv(tables/'rg3_190_gradual_branch_ablation_predictions.csv',index=False,encoding='utf-8-sig')
best=metrics.sort_values(['TP','FP','classifiers'],ascending=[False,True,True]).groupby('classifiers',sort=True).head(1).sort_values('classifiers',ascending=False)
same=metrics[(metrics.TP>=8)&(metrics.FP<=6)].sort_values(['classifiers','FP'])
print(best[['classifiers','TP','FP','FN','TN','F1','candidate']].to_string(index=False))
print('Original performance maintained candidates:')
print(same.to_string(index=False))
text='# RG3 조금씩 줄이기: 기존 구간 예측 기반 가지 제거\n\n'
text+='수정: 이전 판본은 seed列의 anchor 포함 예측을 단독 압력 예측으로 오인해 11개 성능을 과장했다. 이제 score>threshold 및 direction_pass로 압력 단독 예측을 분리했다. 기존 50개의 저장된 구간 제외 예측 TP8/FP6을 정확히 재현했다.\n\n'
text+='분류기 수 | 찾은 가스 | 정상 오탐 | 놓친 가스 | F1\n---:|---:|---:|---:|---:\n'
for _,r in best.iterrows(): text+=f'{r.classifiers} | {r.TP} | {r.FP} | {r.FN} | {r.F1:.3f}\n'
text+='\n## 동일 성능을 유지한 최소 조합\n'
text+=same.head(1).to_string(index=False) if len(same) else '50개 미만에서는 TP8/FP6 유지 조합 없음.'
text+='\n\n## 주의\n이 결과는 이미 본 평가 예측으로 축소 후보를 찾은 개발 분석이다. 독립적인 모델 선택 검증이 아니다. 2개 압력 묶음의 1표/2표 조건도 개발 비교이며 임계값을 재학습하거나 새 자료에서 검증하지 않았다. 기존 모델 파일은 변경하지 않았다.\n분류기 수에는 정상 관계 Ridge 회귀 4개 전처리가 포함되지 않는다. 가지 안의 개별 모델을 제거하는 재학습은 이번 분석에 포함되지 않는다.\n'
(root/'reports/rg3_190_gradual_reduction.md').write_text(text,encoding='utf-8')



 classifiers  TP  FP  FN  TN       F1                                          candidate
          50   8   6   9 560 0.516129 I1_T1_seed42_7_101_seed11_23_37_seed13_29_53_vote2
          41   8   6   9 560 0.516129              I1_T1_seed42_7_101_seed11_23_37_vote1
          38   7   6  10 560 0.466667 I0_T1_seed42_7_101_seed11_23_37_seed13_29_53_vote2
          32   8   6   9 560 0.516129                           I1_T1_seed42_7_101_vote1
          29   7   6  10 560 0.466667              I0_T1_seed42_7_101_seed11_23_37_vote1
          23   7   6  10 560 0.466667                            I1_T1_no_pressure_vote1
          20   7   6  10 560 0.466667                           I0_T1_seed42_7_101_vote1
          14   5   6  12 560 0.357143                            I1_T0_no_pressure_vote1
          11   6   6  11 560 0.413793                            I0_T1_no_pressure_vote1
           2   4   6  13 560 0.296296                            I0_T0_no_pressure_vote1
Original performance 

945